# Return to a conversation and resume it

Run this in a fresh kernel, any time within 60 days of [08-13-01](08-13-01-start-edition-conversations.ipynb).
It starts with the project endpoint and the agent name only. It reads every Foundry conversation id from Foundry.

**Prerequisites:** 08-13-01 has run, and `az login`.

## Step 1: Configuration

Connect to the agent's endpoint.

In [1]:
import os, re, subprocess
from datetime import datetime, timedelta, timezone
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

PROJECT_ENDPOINT = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
AGENT_NAME       = "newsletter-agent"

client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())
agent_client = client.get_openai_client(agent_name=AGENT_NAME)

print(f"Project : {PROJECT_ENDPOINT}")
print(f"Agent   : {AGENT_NAME}")

Project : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent   : newsletter-agent


## Step 2: List the conversations

One row per conversation, with the edition id from its metadata and its age. `expires` is the creation date
plus the documented 60 days; the API does not return it.

In [2]:
def list_conversations():
    """The conversations on this agent's endpoint, one row each: `id` (the Foundry conversation id), `created_at`
    and `metadata`. The openai SDK has no list method, so this is a raw GET."""
    return agent_client.get("/conversations", cast_to=dict, options={"params": {"limit": 100}})["data"]

def show(conversations):
    now = datetime.now(timezone.utc)
    print(f'{"edition id":11} {"published":11} {"age":>8}  {"expires":11} {"last resumed":13} Foundry conversation id')
    for c in sorted(conversations, key=lambda c: c["metadata"]["edition_date"]):
        m, created = c["metadata"], datetime.fromtimestamp(c["created_at"], timezone.utc)
        print(f'{m["edition_id"]:11} {m["edition_date"]:11} {(now - created).days:>3} days  '
              f'{created + timedelta(days=60):%Y-%m-%d}  {m.get("last_resumed", "-"):13} {c["id"]}')

conversations = list_conversations()
show(conversations)

edition id  published        age  expires     last resumed  Foundry conversation id
ed-001      2026-11-16    0 days  2027-01-24  -             conv_0c6a5f3c0806f98f00enUp9LBbfDgBxXLLJBuF7TDBCaS0PBs2
ed-002      2026-11-17    0 days  2027-01-24  -             conv_0ba77b81f77ee14100luVsOsTDPPV4hHcRR6M7MRM0TBgC5NcI
ed-003      2026-11-18    0 days  2027-01-24  -             conv_01aee59eb655757f00dAVT4YtQDPZCYpEpcPA6P9HewjNGjTyE
ed-004      2026-11-19    0 days  2027-01-24  -             conv_0046fb51d8a3fd3900qfWksCLd3qyNSHBLFITshuVenfcAYKEq
ed-005      2026-11-20    0 days  2027-01-24  -             conv_01cd091aec7edaa000GaduHhx7OFwkDlyh2g76FI3bqCo2Occ3


## Step 3: Find the conversation for an edition

Find the conversation whose metadata has the edition id of the oldest edition, and keep its Foundry conversation
id. The API has no metadata filter, so the match is done here.

In [3]:
EDITION_ID = "ed-001"

conversation = next(c for c in conversations if c["metadata"]["edition_id"] == EDITION_ID)
foundry_conv_id = conversation["id"]

print(f'{EDITION_ID}: {conversation["metadata"]["edition_title"]}')
print(foundry_conv_id)

ed-001: Northwind buys into a port, Relecloud heats homes
conv_0c6a5f3c0806f98f00enUp9LBbfDgBxXLLJBuF7TDBCaS0PBs2


## Step 4: Read the history back

List the conversation's items in order: the edition (role `developer`, printed here as its header and one
line per article), then every question and answer so far.

In [4]:
items = agent_client.conversations.items.list(foundry_conv_id, order="asc", limit=100).data

for item in items:
    text = item.content[0].text
    if item.role == "developer":
        lines = text.splitlines()
        text = "\n           ".join(lines[:1] + [line for line in lines if line.startswith("[ed-")])
    print(f"{item.role:>9}: {text}\n")

developer: Contoso Daily Brief, edition ed-001, published 2026-11-16
           [ed-001-01] Business: Northwind Traders buys 40% of the Alderbrook container terminal
           [ed-001-02] Technology: Relecloud's liquid-cooled server hall now heats 2,400 homes
           [ed-001-03] Science: Trey Research finds a kelp forest recovering twice as fast as predicted
           [ed-001-04] Sport: Alpine Ski House wins the opening relay of the Tailspin Cup

     user: My pick from this edition is ed-001-02. In two sentences, why does it matter?

assistant: It matters because Relecloud is showing that a data center can do more than compute: it can also supply useful heat to 2,400 homes, turning waste into a local energy benefit [ed-001-02]. It also cut electricity use by 38% versus an air-cooled hall in the pilot, suggesting a more efficient model that could scale to three more halls in 2027 [ed-001-02].



## Step 5: Resume the conversation

Ask a question that only the stored history can answer, passing the Foundry conversation id. The cell fails if the
answer does not name the article picked in 08-13-01.

In [5]:
first_question = next(item for item in items if item.role == "user").content[0].text
picked_article_id = re.search(r"ed-\d{3}-\d{2}", first_question).group()

response = agent_client.responses.create(
    conversation=foundry_conv_id,
    input="Which article did I pick when we first talked about this edition? Give its article id and headline.",
)
print(response.output_text)

assert picked_article_id in response.output_text, f"the answer does not name {picked_article_id}"

You picked **ed-001-02 — Technology: Relecloud's liquid-cooled server hall now heats 2,400 homes** [ed-001-02].


## Step 6: Update the metadata

Record today's date as `last_resumed` on the conversation, then list again. The API returns no last-activity
time, so metadata is where an application keeps one.

In [6]:
today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
agent_client.conversations.update(foundry_conv_id, metadata={**conversation["metadata"], "last_resumed": today})

show(list_conversations())

edition id  published        age  expires     last resumed  Foundry conversation id
ed-001      2026-11-16    0 days  2027-01-24  2026-11-25    conv_0c6a5f3c0806f98f00enUp9LBbfDgBxXLLJBuF7TDBCaS0PBs2
ed-002      2026-11-17    0 days  2027-01-24  -             conv_0ba77b81f77ee14100luVsOsTDPPV4hHcRR6M7MRM0TBgC5NcI
ed-003      2026-11-18    0 days  2027-01-24  -             conv_01aee59eb655757f00dAVT4YtQDPZCYpEpcPA6P9HewjNGjTyE
ed-004      2026-11-19    0 days  2027-01-24  -             conv_0046fb51d8a3fd3900qfWksCLd3qyNSHBLFITshuVenfcAYKEq
ed-005      2026-11-20    0 days  2027-01-24  -             conv_01cd091aec7edaa000GaduHhx7OFwkDlyh2g76FI3bqCo2Occ3


## Cleanup

Leave the conversations in place to run this notebook again on a later day. To remove them and the agent,
uncomment the lines below.

In [7]:
# for c in list_conversations():
#     agent_client.conversations.delete(c["id"])
# client.agents.delete(agent_name=AGENT_NAME)
# print(f"Deleted the conversations and {AGENT_NAME}")